# Lecture 2: Global Explanations -- Feature Effects
**Methods:** Permutation Feature Importance (PFI), PDP, ICE, ALE.
**Data:** California housing (predict median house value, in $100k) with a gradient-boosting model, plus two small synthetic experiments where we know the ground truth.
Cells marked `TODO` are your tasks.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.inspection import permutation_importance, PartialDependenceDisplay

data = fetch_california_housing(as_frame=True)
X = data.data.sample(6000, random_state=0)
y = data.target.loc[X.index]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0)
model = HistGradientBoostingRegressor(random_state=0).fit(X_train, y_train)
print("R^2 train/test:", model.score(X_train, y_train), model.score(X_test, y_test))
X.describe().T[["mean","std","min","max"]]

## Part 1: Permutation Feature Importance
PFI shuffles one feature and measures how much the model score drops. Use **held-out data**.

In [ ]:
pfi = permutation_importance(model, X_test, y_test, n_repeats=15, random_state=0)
imp = pd.DataFrame({"mean": pfi.importances_mean, "std": pfi.importances_std}, index=X.columns).sort_values("mean")
imp["mean"].plot.barh(xerr=imp["std"], figsize=(6,4), title="PFI on test data (drop in R^2)")
plt.show()

**TODO 1.** Compute PFI on the *training* set and compare with the test-set PFI. Do the rankings differ? When would a large gap signal overfitting?

**TODO 2.** `AveRooms` and `AveBedrms` are correlated. Check the correlation, then permute them *together* as a group (shuffle the two columns with the same row permutation) and compare the group drop with the two individual drops. What does this show about PFI and correlated features?

In [ ]:
# TODO 1
# pfi_train = permutation_importance(model, X_train, y_train, n_repeats=15, random_state=0)

# TODO 2
# print(X[["AveRooms","AveBedrms"]].corr())
# rng = np.random.RandomState(0); Xp = X_test.copy(); perm = rng.permutation(len(Xp))
# Xp[["AveRooms","AveBedrms"]] = X_test[["AveRooms","AveBedrms"]].values[perm]
# group_drop = model.score(X_test, y_test) - model.score(Xp, y_test)

## Part 2: PDP and ICE
PDP = average prediction when feature $x_j$ is set to each grid value. ICE = one curve per instance (the PDP is their average).

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15,4))
PartialDependenceDisplay.from_estimator(model, X_train, ["MedInc","HouseAge","AveOccup"], kind="both",
    subsample=60, random_state=0, ax=ax, ice_lines_kw={"alpha":0.2}, pd_line_kw={"color":"red","lw":3})
plt.show()

**TODO 3.** Interpret the `MedInc` and `HouseAge` curves in plain language. Is the effect monotonic? Where does the ICE spread suggest interactions?

**TODO 4.** Draw a 2D PDP for `Latitude` and `Longitude` (pass `[("Latitude","Longitude")]` and `kind="average"`). What does it show that the 1D plots cannot? Which map of California does it resemble?

In [ ]:
# TODO 3 / 4
# PartialDependenceDisplay.from_estimator(model, X_train, [("Latitude","Longitude")], grid_resolution=25)
# plt.show()

## Part 3: When the PDP hides structure (interactions)
True function: $y = u \cdot z$ with $z \in \{-1, +1\}$. Half the data increases with $u$, half decreases.

In [ ]:
rng = np.random.RandomState(0); n = 1000
u = rng.uniform(-1, 1, n); z = rng.choice([-1, 1], n)
Xi = pd.DataFrame({"u": u, "z": z}); yi = u * z + rng.normal(0, 0.05, n)
rf_i = RandomForestRegressor(200, min_samples_leaf=5, random_state=0).fit(Xi, yi)
PartialDependenceDisplay.from_estimator(rf_i, Xi, ["u"], kind="both", subsample=100, random_state=0,
    ice_lines_kw={"alpha":0.3}, pd_line_kw={"color":"red","lw":3})
plt.show()

**TODO 5.** The PDP is almost flat. Explain why, using the ICE curves. Colour the ICE curves by `z` (compute ICE manually with a loop over a grid) to confirm.

## Part 4: Correlated features -- PDP vs ALE
We implement 1D ALE by hand. Idea: within each quantile bin, average the prediction *difference* between the bin edges using only the data points *in that bin*, then accumulate and centre.

In [ ]:
def ale_1d(model, X, feat, bins=10):
    x = X[feat].values
    edges = np.unique(np.quantile(x, np.linspace(0, 1, bins + 1)))
    idx = np.clip(np.digitize(x, edges[1:-1], right=True), 0, len(edges) - 2)
    eff, cnt = np.zeros(len(edges) - 1), np.zeros(len(edges) - 1)
    for k in range(len(edges) - 1):
        m = idx == k
        if m.sum() == 0: continue
        lo, hi = X[m].copy(), X[m].copy()
        lo[feat], hi[feat] = edges[k], edges[k + 1]
        eff[k] = (model.predict(hi) - model.predict(lo)).mean(); cnt[k] = m.sum()
    ale = np.r_[0, np.cumsum(eff)]
    ale_mid = (ale[:-1] + ale[1:]) / 2
    ale_mid -= np.sum(ale_mid * cnt) / cnt.sum()          # centre
    return (edges[:-1] + edges[1:]) / 2, ale_mid

def pdp_1d(model, X, feat, grid):
    out = []
    for g in grid:
        t = X.copy(); t[feat] = g; out.append(model.predict(t).mean())
    return np.array(out) - np.mean(out)

# x2 is almost a copy of x1; true function y = x1 * x2  (so y ~ x1^2 on the data)
x1 = rng.uniform(0, 1, n); x2 = x1 + rng.normal(0, 0.05, n)
Xc = pd.DataFrame({"x1": x1, "x2": x2}); yc = x1 * x2 + rng.normal(0, 0.02, n)
rf_c = RandomForestRegressor(200, min_samples_leaf=5, random_state=0).fit(Xc, yc)

centers, ale = ale_1d(rf_c, Xc, "x1")
pdp = pdp_1d(rf_c, Xc, "x1", centers)
plt.plot(centers, pdp, label="PDP"); plt.plot(centers, ale, label="ALE")
plt.scatter(Xc["x1"], Xc["x2"] * 0, alpha=0.02, color="k")   # rug
plt.xlabel("x1"); plt.legend(); plt.show()

**TODO 6.** Add the "on-the-data" reference curve $E[y \mid x_1]$ (use `centers**2`, centred the same way) and compare PDP and ALE. Which is closer, and why? Hint: the PDP evaluates the model at $(x_1, x_2)$ combinations such as $(0.1, 0.9)$ that never occur in the data.

**TODO 7.** Apply `ale_1d` to the California model for `MedInc` and `AveRooms` and compare with the PDPs. Where do they disagree?

In [ ]:
# TODO 6 / 7

## Discussion
1. Why is PFI computed on test data, and what goes wrong with correlated features?
2. When is a PDP trustworthy? When would you prefer ALE?
3. A flat PDP: does it mean the feature is unimportant? Give a counter-example from this notebook.
4. Which of these methods show *how* a feature matters and which only show *how much*?